In [ ]:
import os
from pathlib import Path
import shutil

import pandas as pd
import numpy as np
import functools

import teehr

import pyspark as ps
from pyspark.sql import functions as F

from assembly_utils import custom_event_detection as ced 

teehr.__version__

### Setup

In [ ]:
test_eval_dir = Path(Path().cwd(), "FIRO_test_evaluation")

# spark configuration
from teehr.evaluation.spark_session_utils import create_spark_session
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=2,
    executor_memory="50g",
    executor_cores=7
)

# access local evaluation
ev = teehr.LocalReadWriteEvaluation(
    dir_path=test_eval_dir,
    spark=spark
)

ev.enable_logging()

### Implement custom method for ranking detected events

### Top Events Performance Analysis
---------------------------------------------------
- Global Filters/Toggles:
  - Number of largest events (dropdown)
  - Decision variable

#### Top Events Analysis: Heat Map
---------------------------------------------------
- Filters:
  - Mean/Median deterministic forecast
  - Metric (dropdown) -> error, absolute error, bias
---------------------------------------------------
- Heat map:
  - X = event
  - y = lead time
  - color/Z = metric value   

In [ ]:
def create_PF_heatmap(
    quantile: str,
    configuration_toggle: list | str,
    variable_toggle: str
) -> ps.sql.DataFrame:
    '''Creates the heatmap table.'''

    print(f"Beginning calculations for quantile = {quantile}\n")
    
    # define group_by list
    group_by_list = [
        'primary_location_id',
        'configuration_name',
        'variable_name',
        f'event_{quantile}_id',
        'forecast_lead_time_bin'
    ]

    # define metric list
    metric_list = [
        teehr.DeterministicMetrics.RelativeBias(),
        teehr.DeterministicMetrics.RootMeanSquareError(),
        teehr.DeterministicMetrics.PearsonCorrelation()
    ]

    # define filters list
    formatted_configs = ",".join([f"'{x}'" for x in configuration_toggle])
    filter_list = [
         f"configuration_name IN ({formatted_configs})",
         f"variable_name = '{variable_toggle}'",
         f"event_{quantile} = 'true'"
    ]

    # perform aggregation
    sdf = ev.table(
        "joined_timeseries"
    ).filter(
        filter_list
    ).aggregate(
        metrics=metric_list,
        group_by=group_by_list
    ).order_by([
        'primary_location_id',
        f'event_{quantile}_id',
        'forecast_lead_time_bin',
        'configuration_name'
    ]).to_sdf()

    # add threshold column for filtering
    # df['threshold'] = quantile
    sdf = sdf.withColumn('threshold', F.lit(f'{quantile}'))

    # rename event_id column
    # df.rename(columns={f'event_{quantile}_id':'event_id'}, inplace=True)
    sdf = sdf.withColumnRenamed(f'event_{quantile}_id', 'event_id')

    print(f"Finished calculations for quantile = {quantile}\n")

    return sdf


In [ ]:
# assemble heatmap table
quantiles = ['10th', '25th', '50th', '75th', '90th']
configuration_toggle = ["hefs_streamflow_forecast", "benchmark_forecast_hourly_normals"]
variable_toggle = "streamflow_hourly_inst"

result = {}
for quantile in quantiles:
    result[quantile] = create_PF_heatmap(
        quantile=quantile,
        configuration_toggle=configuration_toggle,
        variable_toggle=variable_toggle
    )

# heatmap_df = pd.concat(result.values(), ignore_index=True)
heatmap_sdf = functools.reduce(ps.sql.DataFrame.unionByName, result.values())

In [ ]:
# write to warehouse
ev._write.to_warehouse(source_data=heatmap_sdf, table_name='event_heatmap', write_mode='create_or_replace')

#### Top Events Analysis: Ensemble Performance
---------------------------------------------------
- Filters:
  - Lead time (dropdown or slider)
---------------------------------------------------
- Diagram:
  - box-whisker plot of observed values vs forecast values where box/whisker represents range of member values for a given observed value 

In [ ]:
# N/A -- this can directly query the JTS based on selected event

#### Top Events Analysis: Individual Event Forecast Viewer
---------------------------------------------------
- Filters:
  - Event (dropdown)
  - Lead time (dropdown or slider)
---------------------------------------------------
- Diagram:
  - Plots full ensemble alongside observed for chosen event at selected lead time 


In [ ]:
# N/A -- this can directly query the JTS based on selected event

### Kill Spark

In [ ]:
ev.spark.stop()